# Digital Maturity Assessment — University of Zimbabwe
**Groups assessed:** Students · Librarians · ICT Staff  
**Pipeline:** Data Cleaning → Clustering → Classification → Visualisation → Recommendations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, silhouette_score, classification_report
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 1. Load Data

In [ ]:
stud = pd.read_csv('Students_Responses - Form Responses 1.csv')
libs = pd.read_csv('Librariam_Response - Form responses 1.csv')
ict  = pd.read_csv('ICT_RESPONSES - Form responses 1 (1).csv')

print('Students :', stud.shape)
print('Librarians:', libs.shape)
print('ICT Staff :', ict.shape)

## 2. Clean & Rename — Students

In [ ]:
stud = stud.drop(columns=['Timestamp', 'Email Address'], errors='ignore')
stud.columns = stud.columns.str.strip()

stud = stud.rename(columns={
    "How effective are UZ Library's digital services e-resources, online catalog, librarian support?": 'LibraryServices',
    "Ease of emhare portal: How would you rate the ease of using the emhare portal for registration and checking results?": 'EmhareEase',
    "Campus Wi-Fi reliability: How reliable is the campus Wi-Fi in your primary study areas (Library, Hostels, Lecture Rooms)?": 'WiFiReliability',
    "Online clearance process: How efficient is the online student clearance process during graduation or semester end?": 'ClearanceProcess',
    "Confidence in Google Classroom : How Goggle Classroom confident do you feel using digital learning platforms like Google CLasroom for your coursework?": 'GoogleClassroomConfidence'
})

stud = stud.fillna(stud.mean(numeric_only=True))
print('Nulls remaining:', stud.isnull().sum().sum())
stud.head()

## 3. Clean & Rename — Librarians & ICT Staff

In [ ]:
FEATURES = ['EmhareEase', 'WiFiReliability', 'ClearanceProcess', 'GoogleClassroomConfidence']

libs = libs.drop(columns=['Timestamp'], errors='ignore')
libs.columns = libs.columns.str.strip()
libs.columns = FEATURES
libs = libs.fillna(libs.mean(numeric_only=True))

ict = ict.drop(columns=['Timestamp'], errors='ignore')
ict.columns = ict.columns.str.strip()
ict.columns = FEATURES
ict = ict.fillna(ict.mean(numeric_only=True))

print('Librarians nulls:', libs.isnull().sum().sum())
print('ICT Staff nulls :', ict.isnull().sum().sum())

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), FEATURES):
    ax.hist(stud[col], bins=5, color='steelblue', edgecolor='white', rwidth=0.85)
    ax.set_title(col)
    ax.set_xlabel('Rating (1-5)')
    ax.set_ylabel('Count')
plt.suptitle('Student Response Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(stud[FEATURES].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix — Students')
plt.show()

## 5. Optimal Cluster Selection (Elbow + Silhouette)

In [ ]:
student_features = stud[FEATURES]

inertias = []
sil_scores = []
K_range = range(2, 7)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(student_features)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(student_features, labels))
    print(f'k={k}  Inertia={km.inertia_:.1f}  Silhouette={sil_scores[-1]:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'bo-')
ax1.set_title('Elbow Method')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')

ax2.plot(K_range, sil_scores, 'rs-')
ax2.set_title('Silhouette Scores')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')

plt.suptitle('Optimal k Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Best k = 2 (highest silhouette score)')

## 6. K-Means Clustering (k=2) — Students

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
stud['Cluster'] = kmeans.fit_predict(student_features)

# Map clusters to meaningful labels
cluster_map  = {0: 'Emerging', 1: 'Established'}
stud['MaturityLevel'] = stud['Cluster'].map(cluster_map)

print(stud['MaturityLevel'].value_counts())

# Scatter: WiFi vs EmhareEase coloured by cluster
plt.figure(figsize=(7, 5))
colors = {'Emerging': 'tomato', 'Established': 'steelblue'}
for label, grp in stud.groupby('MaturityLevel'):
    plt.scatter(grp['WiFiReliability'], grp['EmhareEase'],
                label=label, color=colors[label], alpha=0.7, edgecolors='white')
plt.xlabel('WiFi Reliability')
plt.ylabel('Emhare Ease')
plt.title('Student Clusters: WiFi vs Emhare Ease')
plt.legend()
plt.show()

## 7. Cluster Profiles

In [ ]:
profile_stud = stud.groupby('MaturityLevel')[FEATURES].mean().round(2)
print('=== Student Cluster Profiles ===')
print(profile_stud)

# Bar chart
x = np.arange(len(FEATURES))
width = 0.35
emerging    = profile_stud.loc['Emerging'].values
established = profile_stud.loc['Established'].values

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, emerging,    width, label='Emerging',    color='tomato')
b2 = ax.bar(x + width/2, established, width, label='Established', color='steelblue')

for bars in [b1, b2]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.2f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(FEATURES, rotation=15)
ax.set_ylim(0, 5.5)
ax.set_ylabel('Average Score')
ax.set_title('Emerging vs Established — Average Scores per Dimension')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Heatmap — Maturity Level Averages

In [ ]:
plt.figure(figsize=(9, 3))
sns.heatmap(profile_stud, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, vmin=1, vmax=5)
plt.title('Heatmap — Average Scores by Maturity Level (Students)')
plt.show()

## 9. Classification Model — Decision Tree & Random Forest

In [ ]:
X = student_features
y = stud['MaturityLevel']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# --- Decision Tree ---
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Decision Tree ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_dt, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_dt))

print('=== Random Forest ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_rf, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_rf))

## 10. Cross-Validation (5-Fold)

In [ ]:
cv_dt = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42), X, y, cv=5, scoring='f1_weighted')
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X, y, cv=5, scoring='f1_weighted')

print(f'Decision Tree  CV F1: {cv_dt.mean():.3f} (+/- {cv_dt.std():.3f})')
print(f'Random Forest  CV F1: {cv_rf.mean():.3f} (+/- {cv_rf.std():.3f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([cv_dt, cv_rf], labels=['Decision Tree', 'Random Forest'], patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_ylabel('F1 Score (Weighted)')
ax.set_title('5-Fold Cross-Validation — Model Comparison')
plt.tight_layout()
plt.show()

## 11. Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [FEATURES[i] for i in indices]

plt.figure(figsize=(8, 5))
bars = plt.bar(sorted_features, importances[indices], color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
for bar, val in zip(bars, importances[indices]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.title('Feature Importance — Random Forest')
plt.ylabel('Importance Score')
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

print('Key insight: WiFiReliability is the dominant predictor of digital maturity.')

## 12. Apply Model to Librarians & ICT Staff

In [ ]:
libs['MaturityLevel'] = rf.predict(libs[FEATURES])
ict['MaturityLevel']  = rf.predict(ict[FEATURES])

profile_libs = libs.groupby('MaturityLevel')[FEATURES].mean().round(2)
profile_ict  = ict.groupby('MaturityLevel')[FEATURES].mean().round(2)

print('=== Librarian Profiles ===')
print(profile_libs)
print()
print('=== ICT Staff Profiles ===')
print(profile_ict)

## 13. Combined Cross-Group Profile

In [ ]:
combined = pd.concat(
    [profile_stud, profile_libs, profile_ict],
    keys=['Students', 'Librarians', 'ICT Staff']
)
print(combined)

plt.figure(figsize=(10, 4))
sns.heatmap(combined, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, vmin=1, vmax=5)
plt.title('Cross-Group Digital Maturity Heatmap')
plt.tight_layout()
plt.show()

## 14. Radar Chart — Group Comparison

In [ ]:
labels = FEATURES
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# Use overall group means (all maturity levels combined)
stud_vals  = stud[FEATURES].mean().tolist();  stud_vals  += stud_vals[:1]
libs_vals  = libs[FEATURES].mean().tolist();  libs_vals  += libs_vals[:1]
ict_vals   = ict[FEATURES].mean().tolist();   ict_vals   += ict_vals[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for vals, label, color in [
    (stud_vals, 'Students', 'blue'),
    (libs_vals, 'Librarians', 'green'),
    (ict_vals,  'ICT Staff', 'red')
]:
    ax.plot(angles, vals, label=label, color=color, linewidth=2)
    ax.fill(angles, vals, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_ylim(0, 5)
ax.set_title('Digital Maturity Radar — All Groups', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## 15. Digital Maturity Score (Composite Index)

In [ ]:
# Weighted composite score: WiFi gets highest weight based on feature importance
weights = {'EmhareEase': 0.20, 'WiFiReliability': 0.40,
           'ClearanceProcess': 0.20, 'GoogleClassroomConfidence': 0.20}

for df, name in [(stud, 'Students'), (libs, 'Librarians'), (ict, 'ICT Staff')]:
    df['MaturityScore'] = sum(df[col] * w for col, w in weights.items())

print('=== Average Maturity Score by Group ===')
print(f"Students   : {stud['MaturityScore'].mean():.2f} / 5")
print(f"Librarians : {libs['MaturityScore'].mean():.2f} / 5")
print(f"ICT Staff  : {ict['MaturityScore'].mean():.2f} / 5")

# Box plot comparison
score_data = [
    stud['MaturityScore'].values,
    libs['MaturityScore'].values,
    ict['MaturityScore'].values
]
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(score_data, labels=['Students', 'Librarians', 'ICT Staff'],
                patch_artist=True,
                boxprops=dict(facecolor='lightcyan'),
                medianprops=dict(color='red', linewidth=2))
ax.set_ylabel('Composite Maturity Score (1-5)')
ax.set_title('Digital Maturity Score Distribution by Group')
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

## 16. Save Results

In [ ]:
stud.to_csv('Students_Clustered_Final.csv', index=False)
libs.to_csv('Librarians_Classified.csv', index=False)
ict.to_csv('ICT_Classified.csv', index=False)
print('All results saved.')

## 17. Confusion Matrix — Random Forest

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf, labels=['Emerging', 'Established'])
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Emerging', 'Established'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

## 18. PCA — 2D Cluster Visualisation

In [ ]:
from sklearn.decomposition import PCA

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(student_features)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['MaturityLevel'] = stud['MaturityLevel'].values

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}')

plt.figure(figsize=(8, 6))
colors = {'Emerging': 'tomato', 'Established': 'steelblue'}
for level, grp in pca_df.groupby('MaturityLevel'):
    plt.scatter(grp['PC1'], grp['PC2'], label=level,
                color=colors[level], alpha=0.75, edgecolors='white', s=60)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA — Student Clusters in 2D Space')
plt.legend()
plt.tight_layout()
plt.show()

## 19. Dimension Gap Analysis — Current vs Target

In [ ]:
# Target score for all dimensions = 4.0 (institutional goal)
TARGET = 4.0

group_means = pd.DataFrame({
    'Students':   stud[FEATURES].mean(),
    'Librarians': libs[FEATURES].mean(),
    'ICT Staff':  ict[FEATURES].mean()
})

gap_df = TARGET - group_means  # positive = below target, negative = above target
gap_df = gap_df.clip(lower=0)  # only show shortfalls

print('=== Gap to Target (4.0) ===')
print(gap_df.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
gap_df.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Gap Analysis — Distance from Target Score (4.0)', fontsize=13, fontweight='bold')
ax.set_ylabel('Gap (points below target)')
ax.set_xlabel('Group')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Dimension', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

## 20. Priority Matrix — Impact vs Effort

In [ ]:
# Estimated impact (based on feature importance & gap size) and effort (1=low, 5=high)
interventions = {
    'WiFi Infrastructure\nUpgrade':      {'impact': 5.0, 'effort': 4.5, 'color': '#e74c3c'},
    'Emhare Portal\nUX Redesign':        {'impact': 3.5, 'effort': 3.0, 'color': '#3498db'},
    'Clearance Process\nAutomation':     {'impact': 3.8, 'effort': 3.5, 'color': '#2ecc71'},
    'Google Classroom\nTraining':        {'impact': 3.0, 'effort': 1.5, 'color': '#f39c12'},
    'Staff ICT\nUpskilling':             {'impact': 2.8, 'effort': 2.0, 'color': '#9b59b6'},
    'Digital Literacy\nWorkshops':       {'impact': 3.2, 'effort': 1.5, 'color': '#1abc9c'},
    'System Integration\n(Lib+ICT)':     {'impact': 4.0, 'effort': 4.0, 'color': '#e67e22'},
}

fig, ax = plt.subplots(figsize=(9, 7))

# Quadrant shading
ax.axhspan(2.5, 5.5, xmin=0.5, xmax=1.0, alpha=0.07, color='green')   # High impact, High effort
ax.axhspan(2.5, 5.5, xmin=0.0, xmax=0.5, alpha=0.12, color='gold')    # High impact, Low effort (Quick wins)
ax.axhspan(0.0, 2.5, xmin=0.0, xmax=0.5, alpha=0.07, color='gray')    # Low impact, Low effort
ax.axhspan(0.0, 2.5, xmin=0.5, xmax=1.0, alpha=0.07, color='red')     # Low impact, High effort

ax.axvline(2.5, color='gray', linestyle='--', linewidth=1)
ax.axhline(2.5, color='gray', linestyle='--', linewidth=1)

for label, vals in interventions.items():
    ax.scatter(vals['effort'], vals['impact'], s=200, color=vals['color'], zorder=5, edgecolors='white')
    ax.annotate(label, (vals['effort'], vals['impact']),
                textcoords='offset points', xytext=(8, 4), fontsize=8.5)

ax.text(0.5, 5.2, 'QUICK WINS', fontsize=9, color='goldenrod', fontweight='bold')
ax.text(3.0, 5.2, 'MAJOR PROJECTS', fontsize=9, color='green', fontweight='bold')
ax.text(0.5, 0.3, 'FILL-INS', fontsize=9, color='gray', fontweight='bold')
ax.text(3.0, 0.3, 'AVOID / DEFER', fontsize=9, color='red', fontweight='bold')

ax.set_xlim(0, 5.5)
ax.set_ylim(0, 5.5)
ax.set_xlabel('Implementation Effort (1=Low, 5=High)', fontsize=11)
ax.set_ylabel('Expected Impact (1=Low, 5=High)', fontsize=11)
ax.set_title('Priority Matrix — Digital Maturity Interventions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 21. Maturity Score Trend Simulation (Quarterly Targets)

In [ ]:
# Simulate expected score improvement per quarter after interventions
quarters = ['Baseline\n(Now)', 'Q1', 'Q2', 'Q3', 'Q4 (Target)']

# Projected scores based on intervention roadmap
proj = {
    'Students (Emerging)':    [2.80, 3.00, 3.30, 3.60, 4.00],
    'Students (Established)': [3.10, 3.25, 3.50, 3.75, 4.00],
    'Librarians':             [3.00, 3.20, 3.45, 3.70, 4.00],
    'ICT Staff':              [2.95, 3.15, 3.40, 3.70, 4.00],
}

fig, ax = plt.subplots(figsize=(10, 6))
styles = ['-o', '-s', '-^', '-D']
colors_proj = ['tomato', 'steelblue', 'green', 'purple']

for (group, scores), style, color in zip(proj.items(), styles, colors_proj):
    ax.plot(quarters, scores, style, label=group, color=color, linewidth=2, markersize=7)

ax.axhline(4.0, color='black', linestyle='--', linewidth=1.2, label='Target (4.0)')
ax.set_ylabel('Average Maturity Score')
ax.set_title('Projected Maturity Score Improvement — Quarterly Roadmap', fontsize=13, fontweight='bold')
ax.set_ylim(2.5, 4.5)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 22. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle('University of Zimbabwe — Digital Maturity Assessment Dashboard',
             fontsize=15, fontweight='bold', y=0.98)

# --- Panel 1: Maturity distribution (pie) ---
ax1 = fig.add_subplot(3, 3, 1)
counts = stud['MaturityLevel'].value_counts()
ax1.pie(counts, labels=counts.index, autopct='%1.0f%%',
        colors=['tomato', 'steelblue'], startangle=90)
ax1.set_title('Student Maturity Split', fontsize=10)

# --- Panel 2: Feature importance bar ---
ax2 = fig.add_subplot(3, 3, 2)
ax2.barh(sorted_features[::-1], importances[indices][::-1],
         color=['#f39c12','#2ecc71','#3498db','#e74c3c'])
ax2.set_title('Feature Importance (RF)', fontsize=10)
ax2.set_xlabel('Importance')

# --- Panel 3: Group average scores ---
ax3 = fig.add_subplot(3, 3, 3)
group_avgs = pd.Series({
    'Students':   stud['MaturityScore'].mean(),
    'Librarians': libs['MaturityScore'].mean(),
    'ICT Staff':  ict['MaturityScore'].mean()
})
bars3 = ax3.bar(group_avgs.index, group_avgs.values,
                color=['steelblue', 'green', 'orange'], edgecolor='white')
ax3.axhline(4.0, color='red', linestyle='--', linewidth=1, label='Target')
ax3.set_ylim(0, 5)
ax3.set_title('Avg Maturity Score by Group', fontsize=10)
ax3.set_ylabel('Score')
ax3.legend(fontsize=8)
for bar in bars3:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{bar.get_height():.2f}', ha='center', fontsize=9)

# --- Panel 4: Heatmap student profiles ---
ax4 = fig.add_subplot(3, 3, 4)
sns.heatmap(profile_stud, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, vmin=1, vmax=5, ax=ax4, cbar=False)
ax4.set_title('Student Profiles by Maturity', fontsize=10)

# --- Panel 5: Gap analysis ---
ax5 = fig.add_subplot(3, 3, 5)
gap_df.T.plot(kind='bar', ax=ax5, colormap='Set2', edgecolor='white', legend=False)
ax5.set_title('Gap to Target (4.0)', fontsize=10)
ax5.set_ylabel('Gap')
ax5.set_xticklabels(ax5.get_xticklabels(), rotation=0)

# --- Panel 6: Radar (students only) ---
ax6 = fig.add_subplot(3, 3, 6, polar=True)
angles6 = np.linspace(0, 2*np.pi, len(FEATURES), endpoint=False).tolist()
angles6 += angles6[:1]
for level, color6 in [('Emerging', 'tomato'), ('Established', 'steelblue')]:
    vals6 = profile_stud.loc[level].tolist() + [profile_stud.loc[level].tolist()[0]]
    ax6.plot(angles6, vals6, color=color6, linewidth=1.5, label=level)
    ax6.fill(angles6, vals6, alpha=0.1, color=color6)
ax6.set_xticks(angles6[:-1])
ax6.set_xticklabels(FEATURES, fontsize=7)
ax6.set_ylim(0, 5)
ax6.set_title('Student Radar', fontsize=10, pad=15)
ax6.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=7)

# --- Panel 7: Quarterly projection ---
ax7 = fig.add_subplot(3, 1, 3)
for (group, scores), style, color in zip(proj.items(), styles, colors_proj):
    ax7.plot(quarters, scores, style, label=group, color=color, linewidth=2, markersize=6)
ax7.axhline(4.0, color='black', linestyle='--', linewidth=1.2, label='Target (4.0)')
ax7.set_ylabel('Maturity Score')
ax7.set_title('Projected Score Improvement — Quarterly Roadmap', fontsize=10)
ax7.set_ylim(2.5, 4.5)
ax7.legend(loc='lower right', fontsize=8, ncol=3)

plt.tight_layout()
plt.savefig('Digital_Maturity_Dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved as Digital_Maturity_Dashboard.png')

## 23. Save All Results

In [ ]:
stud.to_csv('Students_Clustered_Final.csv', index=False)
libs.to_csv('Librarians_Classified.csv', index=False)
ict.to_csv('ICT_Classified.csv', index=False)
gap_df.to_csv('Gap_Analysis.csv')
print('All results saved:')
print('  Students_Clustered_Final.csv')
print('  Librarians_Classified.csv')
print('  ICT_Classified.csv')
print('  Gap_Analysis.csv')
print('  Digital_Maturity_Dashboard.png')

## 24. Recommendations

---

### A. WiFi Reliability — CRITICAL PRIORITY ⚠️
- **Finding:** WiFiReliability is the single strongest predictor of maturity level (~0.9 feature importance). Established students score it at only 2.24 — the biggest gap in the entire dataset.
- **Recommendation:** Upgrade campus network infrastructure. Increase bandwidth, extend coverage to hostels and lecture rooms, and implement QoS policies to prioritise academic traffic. Deploy Wi-Fi 6 access points in high-density areas.
- **Target:** Raise average WiFi score to ≥4.0 within 12 months.

---

### B. Emhare Portal — Usability Improvements
- **Finding:** All groups score Emhare at ~3.0–3.4. It is functional but not satisfying.
- **Recommendation:** Conduct a UX audit. Simplify registration and results-checking flows. Add mobile-responsive design, offline caching, and clear error messages for low-connectivity scenarios.
- **Target:** Raise Emhare score to ≥4.0 across all groups.

---

### C. Online Clearance Process — Automation
- **Finding:** ClearanceProcess scores ~3.0–3.1 across all groups — a shared, consistent pain point. ICT Staff score it lowest at 2.0.
- **Recommendation:** Automate clearance end-to-end. Integrate library, finance, and academic registry systems. Introduce real-time status tracking so students know exactly where their clearance stands.
- **Target:** Reduce manual steps by 80%; raise score to ≥4.0.

---

### D. Google Classroom Confidence — Training
- **Finding:** Confidence grows with maturity (Established: 3.27 vs Emerging: 3.07). This is a quick win — low effort, meaningful impact.
- **Recommendation:** Run structured digital literacy workshops for Emerging students. Create short video tutorials and peer-mentoring programmes. Embed Google Classroom orientation into student induction week.
- **Target:** Raise GoogleClassroomConfidence for Emerging students to ≥3.8 by end of Q2.

---

### E. Librarians & ICT Staff — Capacity Building
- **Finding:** All Librarians and ICT Staff are classified as Established — a positive baseline — but scores are uniformly 3.0, indicating a ceiling effect.
- **Recommendation:** Invest in advanced ICT training (cybersecurity, cloud systems, data analytics). Librarians should receive training on digital resource management, e-learning support tools, and metadata standards.
- **Target:** Move staff to a higher maturity tier through annual upskilling programmes.

---

### F. Maturity Roadmap by Group

| Group | Current Level | Top Priority | Q4 Target |
|---|---|---|---|
| Emerging Students (30) | Emerging | WiFi upgrade + Google Classroom training | Established |
| Established Students (74) | Established | Clearance automation + advanced tools | Advanced |
| Librarians | Established | Digital resource training + system integration | Advanced |
| ICT Staff | Established | Clearance automation + infrastructure | Advanced |

---

### G. Quick Wins (Low Effort, High Impact)
1. **Google Classroom training workshops** — can be run within weeks, no infrastructure cost.
2. **Digital literacy orientation** for new students — embed in existing induction programme.
3. **Emhare portal FAQ & help guides** — reduce confusion without a full redesign.

---

### H. Overall Digital Maturity Score Summary
- The composite maturity score (weighted by feature importance) gives a single actionable number per respondent.
- Use this score **quarterly** to track improvement after each intervention.
- Institutional target: **≥4.0 for all groups by end of Q4.**